# IIT-07 — Coupe et décomposabilité (marche 1 de l'escalier `iit_lean`)

Ce carnet est le **premier consommateur pédagogique** du lake [`iit_lean`](../SymbolicAI/Lean/iit_lean/README.md) — la **moitié formelle** de la veine ouverte par le notebook 06 : ce que l'échange Aaronson/Tononi mesure sur des instances numériques (Φ_DM sur F_p, grilles PyPhi), le lake le formalise en théorèmes rendus par le noyau Lean. Il est la **marche 1** d'un escalier de trois marches défini par la [note de conception](../../docs/lean/iit-lean-escalier.md) : le vocabulaire de la coupe et le théorème A.

L'objet final de la veine — un système **intégré pour toute coupe** à **comportement trivial** — n'apparaît pas ici : il est le théorème B de la marche 2. Ce que cette marche paie, c'est le prix d'entrée : dire précisément **ce que « couper un système » veut dire**, et prouver la direction « facile » — un système dont les moitiés n'interagissent pas est décomposable.

## 0. Ce que ce carnet fait — et ne fait pas

**Fait** : charger `IIT.Integration` (19 déclarations, 0 `sorry`, sans Mathlib), relire chaque définition du vocabulaire de coupe sur des **instances calculées** (`#eval` sur `Fin`, `Bool`), construire un **oracle exécutable** de l'indépendance de coupe par énumération finie, et faire rendre au noyau le théorème A (`cutIndep_of_sides_independent`) ainsi que son application à l'identité — avec les axiomes exacts affichés par `#print axioms`.

**Ne fait pas** : aucune valeur numérique de Φ (le lake formalise un **proxy relationnel** de décomposition, pas Φ — le grade honnête est rappelé en §10) ; aucune preuve du théorème B ni du théorème C (marches 2 et 3 de l'escalier) ; aucune dépendance Mathlib — tout vit dans le noyau et les `List`/`Bool` du core.

## 1. Import du module

In [1]:
import IIT.Integration
open IIT

-- Le theoricien A du lake, charge et verifie par le noyau :
#check @cutIndep_of_sides_independent
#print axioms cutIndep_of_sides_independent

import IIT.Integration
open IIT

-- Le theoricien A du lake, charge et verifie par le noyau :
#check @cutIndep_of_sides_independent
──────▶  @cutIndep_of_sides_independent : ∀ {n : Nat} (c : Cut n) (f : System n),
  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →
    (∀ (s s' : State n),
        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →
      cutIndep c f
#print axioms cutIndep_of_sides_independent
──────▶  'IIT.cutIndep_of_sides_independent' depends on axioms: [propext, Quot.sound]
--% env 0

Raw input:
{"cmd": "import IIT.Integration\nopen IIT\n\n-- Le theoricien A du lake, charge et verifie par le noyau :\n#check @cutIndep_of_sides_independent\n#print axioms cutIndep_of_sides_independent"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "@cutIndep_of_sides_independent : ∀ {n : Nat} (c : Cut n) (f : System n),\n  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →\n    (∀ (s s' : State n),\n        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →\n      cutIndep c f"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "'IIT.cutIndep_of_sides_independent' depends on axioms: [propext, Quot.sound]"}],
 "env": 0}

L'import passe : le lake `iit_lean` est construit sur cette machine, et le théorème A est rendu par le noyau — la liste exacte de ses axiomes est dans la sortie ci-dessus (aucun `sorry`, aucun axiome ajouté par le lake). Le jumeau i18n `IIT.Integration_en` (namespace `IIT_en`, byte-identique hors docstrings) existe dans le même module ; ce carnet consomme le module FR.

## 2. Un état, un système

Le lake pose un cadre minimaliste : un système déterministe à `n` éléments booléens.

- un **état** est une valuation booléenne des éléments : `State n := Fin n → Bool` — une fonction qui à chaque indice `i < n` associe un bit ;
- un **système** est la règle d'évolution, une fonction d'états : `System n := State n → State n`.

C'est volontairement plus pauvre qu'un réseau de PyPhi (pas de probabilités, pas de TPM) : la question n'est pas « combien vaut l'intégration » mais « **existe-t-il** une coupe qui dissolve le système ».

In [2]:
-- Trois etats d'un systeme a 3 elements, notes bit par bit.
def sOneHot : State 3 := fun i => decide (i.val = 0)   -- un seul bit vrai (indice 0)
def sTrue   : State 3 := fun _ => true                  -- tout vrai
def sMix    : State 3 := fun i => decide (i.val = 2)    -- un seul bit vrai (indice 2)

-- Pour lire un etat comme un triplet de bits :
def bits3 (s : State 3) : Bool × Bool × Bool := (s 0, s 1, s 2)

#eval bits3 sOneHot
#eval bits3 sTrue
#eval bits3 sMix

-- Deux systemes a 3 elements :
def identS : System 3 := fun s => s                                   -- l'identite
def maj3   : System 3 := fun s =>                                     -- la majorite globale
  fun _ => (s 0 && s 1) || (s 0 && s 2) || (s 1 && s 2)

#eval (bits3 (identS sMix), bits3 (maj3 sOneHot), bits3 (maj3 sMix))

-- Trois etats d'un systeme a 3 elements, notes bit par bit.
def sOneHot : State 3 := fun i => decide (i.val = 0)   -- un seul bit vrai (indice 0)
def sTrue   : State 3 := fun _ => true                  -- tout vrai
def sMix    : State 3 := fun i => decide (i.val = 2)    -- un seul bit vrai (indice 2)

-- Pour lire un etat comme un triplet de bits :
def bits3 (s : State 3) : Bool × Bool × Bool := (s 0, s 1, s 2)

#eval bits3 sOneHot
─────▶  (true, false, false)
#eval bits3 sTrue
─────▶  (true, true, true)
#eval bits3 sMix
─────▶  (false, false, true)

-- Deux systemes a 3 elements :
def identS : System 3 := fun s => s                                   -- l'identite
def maj3   : System 3 := fun s =>                                     -- la majorite globale
  fun _ => (s 0 && s 1) || (s 0 && s 2) || (s 1 && s 2)

#eval (bits3 (identS sMix), bits3 (maj3 sOneHot), bits3 (maj3 sMix))
─────▶  ((false, false, true), (false, false, false), false, false, false)
--% env 1

Raw input:
{"cmd": "-- Trois etats d'un systeme a 3 elements, notes bit par bit.\ndef sOneHot : State 3 := fun i => decide (i.val = 0)   -- un seul bit vrai (indice 0)\ndef sTrue   : State 3 := fun _ => true                  -- tout vrai\ndef sMix    : State 3 := fun i => decide (i.val = 2)    -- un seul bit vrai (indice 2)\n\n-- Pour lire un etat comme un triplet de bits :\ndef bits3 (s : State 3) : Bool \u00d7 Bool \u00d7 Bool := (s 0, s 1, s 2)\n\n#eval bits3 sOneHot\n#eval bits3 sTrue\n#eval bits3 sMix\n\n-- Deux systemes a 3 elements :\ndef identS : System 3 := fun s => s                                   -- l'identite\ndef maj3   : System 3 := fun s =>                                     -- la majorite globale\n  fun _ => (s 0 && s 1) || (s 0 && s 2) || (s 1 && s 2)\n\n#eval (bits3 (identS sMix), bits3 (maj3 sOneHot), bits3 (maj3 sMix))", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 5},
   "data": "(true, false, false)"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 5},
   "data": "(true, true, true)"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 5},
   "data": "(false, false, true)"},
  {"severity": "info",
   "pos": {"line": 18, "column": 0},
   "endPos": {"line": 18, "column": 5},
   "data":
   "((false, false, true), (false, false, false), false, false, false)"}],
 "env": 1}

L'identité recopie l'état ; la majorité **globalise** — chaque élément suivant prend le vote des trois, si bien qu'aucun sous-ensemble ne suffit à prédire un bit de sortie. Entre ces deux extrêmes, la question de la coupe se posera très différemment.

## 3. Une coupe non triviale

Couper un système, c'est partager ses éléments en deux côtés **non vides** — la structure `Cut` porte ce partage comme un sélecteur booléen plus deux témoins d'existence. La non-trivialité n'est pas un détail : sans elle, « tout d'un côté, rien de l'autre » serait une coupe, et tout système serait décomposable en soi-même plus rien.

In [3]:
-- La coupe {0} | {1,2} sur un systeme a 3 elements :
def cutSolo : Cut 3 where
  sel i := decide (i.val = 0)
  ha := ⟨0, by decide⟩   -- le cote A contient 0
  hb := ⟨1, by decide⟩   -- le cote B contient 1

-- Le selecteur, lu sur les trois indices :
#eval (cutSolo.sel 0, cutSolo.sel 1, cutSolo.sel 2)

-- La coupe {0} | {1,2} sur un systeme a 3 elements :
def cutSolo : Cut 3 where
  sel i := decide (i.val = 0)
  ha := ⟨0, by decide⟩   -- le cote A contient 0
  hb := ⟨1, by decide⟩   -- le cote B contient 1

-- Le selecteur, lu sur les trois indices :
#eval (cutSolo.sel 0, cutSolo.sel 1, cutSolo.sel 2)
─────▶  (true, false, false)
--% env 2

Raw input:
{"cmd": "-- La coupe {0} | {1,2} sur un systeme a 3 elements :\ndef cutSolo : Cut 3 where\n  sel i := decide (i.val = 0)\n  ha := \u27e80, by decide\u27e9   -- le cote A contient 0\n  hb := \u27e81, by decide\u27e9   -- le cote B contient 1\n\n-- Le selecteur, lu sur les trois indices :\n#eval (cutSolo.sel 0, cutSolo.sel 1, cutSolo.sel 2)", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "(true, false, false)"}],
 "env": 2}

`sel i = true` met l'élément `i` côté A, `false` côté B — ici l'élément 0 seul contre les éléments 1 et 2. Les champs `ha` et `hb` sont les **preuves** que chaque côté porte au moins un élément : la non-trivialité est un théorème miniature intégré à la donnée, pas une convention d'honnêteté laissée au lecteur.

## 4. Le motif d'un côté — `projT` et `projF`

Pour parler de ce qu'un côté « voit » de l'état, le lake définit les **motifs** : la projection qui préserve les bits du côté considéré et met les autres à `false`.

In [4]:
-- Les deux motifs de l'etat tout-vrai, pour la coupe {0} | {1,2} :
#eval (bits3 (projT cutSolo sTrue), bits3 (projF cutSolo sTrue))

#eval (bits3 (projT cutSolo sOneHot), bits3 (projF cutSolo sOneHot))

-- Les deux motifs de l'etat tout-vrai, pour la coupe {0} | {1,2} :
#eval (bits3 (projT cutSolo sTrue), bits3 (projF cutSolo sTrue))
─────▶  ((true, false, false), false, true, true)

#eval (bits3 (projT cutSolo sOneHot), bits3 (projF cutSolo sOneHot))
─────▶  ((true, false, false), false, false, false)
--% env 3

Raw input:
{"cmd": "-- Les deux motifs de l'etat tout-vrai, pour la coupe {0} | {1,2} :\n#eval (bits3 (projT cutSolo sTrue), bits3 (projF cutSolo sTrue))\n\n#eval (bits3 (projT cutSolo sOneHot), bits3 (projF cutSolo sOneHot))", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "((true, false, false), false, true, true)"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "((true, false, false), false, false, false)"}],
 "env": 3}

Le motif côté A de `sTrue` est `(true, false, false)` : le bit de l'élément 0 est préservé, les bits 1 et 2 sont masqués. Deux états ont le même motif côté A exactement lorsqu'ils coïncident sur le côté A — le motif est la trace d'un côté sur l'état global.

## 5. Quand une coupe est-elle indépendante ? — `cutIndep` relu

La définition du lake :

$$\texttt{cutIndep } c\ f \;:\Leftrightarrow\; \forall\, x\, y,\ \exists\, z,\quad \texttt{projT}\ c\ (f\ z) = \texttt{projT}\ c\ (f\ x)\ \wedge\ \texttt{projF}\ c\ (f\ z) = \texttt{projF}\ c\ (f\ y)$$

Lecture : **tout couple de motifs réalisables séparément** (le motif A de `x`, le motif B de `y`) **est réalisable ensemble** (par un certain `z`). Autrement dit : l'image du système contient le produit des images de ses deux parties — la coupe ne fait rien perdre, elle découpe sans détruire, le système « ne tient pas » à travers elle.

Comme tout est fini ici, cette définition est **exécutable** : on énumère tous les états, on teste tous les couples `(x, y)` et tous les candidats `z`. Construisons cet oracle — il servira trois fois (ici, au §8, et à l'exercice 3).

In [5]:
-- Enumeration de tous les etats d'un systeme a n elements (2^n etats).
def stateOfNat (n v : Nat) : State n := fun i => ((v >>> i.val) % 2) == 1
def allStates (n : Nat) : List (State n) := (List.range (2 ^ n)).map (stateOfNat n)

-- Oracle : la definition cutIndep, testee par enumeration exhaustive (n = 3).
def cutIndep3 (c : Cut 3) (f : System 3) : Bool :=
  (allStates 3).all fun x =>
    (allStates 3).all fun y =>
      (allStates 3).any fun z =>
        bits3 (projT c (f z)) == bits3 (projT c (f x))
        && bits3 (projF c (f z)) == bits3 (projF c (f y))

#eval (allStates 3).map (fun s => bits3 s)   -- les 8 etats, pour l'oeil
#eval cutIndep3 cutSolo identS               -- l'identite est-elle coupable ?

-- Enumeration de tous les etats d'un systeme a n elements (2^n etats).
def stateOfNat (n v : Nat) : State n := fun i => ((v >>> i.val) % 2) == 1
def allStates (n : Nat) : List (State n) := (List.range (2 ^ n)).map (stateOfNat n)

-- Oracle : la definition cutIndep, testee par enumeration exhaustive (n = 3).
def cutIndep3 (c : Cut 3) (f : System 3) : Bool :=
  (allStates 3).all fun x =>
    (allStates 3).all fun y =>
      (allStates 3).any fun z =>
        bits3 (projT c (f z)) == bits3 (projT c (f x))
        && bits3 (projF c (f z)) == bits3 (projF c (f y))

#eval (allStates 3).map (fun s => bits3 s)   -- les 8 etats, pour l'oeil
─────▶  [(false, false, false),
 (true, false, false),
 (false, true, false),
 (true, true, false),
 (false, false, true),
 (true, false, true),
 (false, true, true),
 (true, true, true)]
#eval cutIndep3 cutSolo identS               -- l'identite est-elle coupable ?
─────▶  true
--% env 4

Raw input:
{"cmd": "-- Enumeration de tous les etats d'un systeme a n elements (2^n etats).\ndef stateOfNat (n v : Nat) : State n := fun i => ((v >>> i.val) % 2) == 1\ndef allStates (n : Nat) : List (State n) := (List.range (2 ^ n)).map (stateOfNat n)\n\n-- Oracle : la definition cutIndep, testee par enumeration exhaustive (n = 3).\ndef cutIndep3 (c : Cut 3) (f : System 3) : Bool :=\n  (allStates 3).all fun x =>\n    (allStates 3).all fun y =>\n      (allStates 3).any fun z =>\n        bits3 (projT c (f z)) == bits3 (projT c (f x))\n        && bits3 (projF c (f z)) == bits3 (projF c (f y))\n\n#eval (allStates 3).map (fun s => bits3 s)   -- les 8 etats, pour l'oeil\n#eval cutIndep3 cutSolo identS               -- l'identite est-elle coupable ?", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 13, "column": 0},
   "endPos": {"line": 13, "column": 5},
   "data":
   "[(false, false, false),\n (true, false, false),\n (false, true, false),\n (true, true, false),\n (false, false, true),\n (true, false, true),\n (false, true, true),\n (true, true, true)]"},
  {"severity": "info",
   "pos": {"line": 14, "column": 0},
   "endPos": {"line": 14, "column": 5},
   "data": "true"}],
 "env": 4}

L'oracle rend `true` pour l'identité : chaque motif côté A `(b, false, false)` est réalisé par 4 états, chaque motif côté B `(false, b, b')` par 2 — le produit `2 × 4 = 8` tient tout entier dans l'image (qui compte exactement 8 états). La coupe `{0} | {1,2}` découpe l'identité **sans rien perdre** : elle est indépendante.

## 6. Décomposable, intégré — et le piège du quantificateur

Sur cette base, deux mots :

- `decomposable f` := **il existe** une coupe indépendante — un seul témoin suffit ;
- `integrated f` := **aucune** coupe n'est indépendante — la négation du précédent.

Le piège : prouver « décomposable » et prouver « intégré » n'ont **pas le même prix**. Le premier se paie par un témoin ; le second exige de fermer **toutes** les coupes d'un coup — pour tout `n`, il y en a `2^n − 2`. C'est toute la différence entre l'oracle du §5 (qui teste **une** coupe) et le théorème B de la marche 2 (qui les ferme **toutes** — voir le teaser du §8).

## 7. Le théorème A et son témoin constructif

Le théorème A dit : si chaque côté du système **n'influence que lui-même** — les bits de sortie côté A ne dépendent que des bits d'entrée côté A (`hA`), idem côté B (`hB`) — alors la coupe est indépendante. La preuve est un **témoin constructif** : le `z` cherché est le **mélange** des entrées, `fun i => if c.sel i then x i else y i` — les bits de `x` côté A, ceux de `y` côté B. La localité garantit que chaque côté produit exactement son motif attendu.

Le lake le dit en deux noms : `cutIndep_of_sides_independent` (l'énoncé travaillé) et `product_decomposable` (l'alias sous le **nom figé** par l'issue #18599 — un contrat de nommage, pour qu'un énoncé référencé reste trouvable).

In [6]:
-- Application travaille : l'identite est decomposable, par le theoreme A.
-- Les deux hypotheses de localite sont triviales pour l'identite : hag i hi
-- dit exactement ce qu'il faut conclure.
theorem ident3_decomposable : decomposable (fun s : State 3 => s) :=
  product_decomposable cutSolo (fun s => s)
    (fun _ _ hag i hi => hag i hi) (fun _ _ hag i hi => hag i hi)

#print axioms ident3_decomposable

-- Application travaille : l'identite est decomposable, par le theoreme A.
-- Les deux hypotheses de localite sont triviales pour l'identite : hag i hi
-- dit exactement ce qu'il faut conclure.
theorem ident3_decomposable : decomposable (fun s : State 3 => s) :=
  product_decomposable cutSolo (fun s => s)
    (fun _ _ hag i hi => hag i hi) (fun _ _ hag i hi => hag i hi)

#print axioms ident3_decomposable
──────▶  'ident3_decomposable' depends on axioms: [propext, Quot.sound]
--% env 5

Raw input:
{"cmd": "-- Application travaille : l'identite est decomposable, par le theoreme A.\n-- Les deux hypotheses de localite sont triviales pour l'identite : hag i hi\n-- dit exactement ce qu'il faut conclure.\ntheorem ident3_decomposable : decomposable (fun s : State 3 => s) :=\n  product_decomposable cutSolo (fun s => s)\n    (fun _ _ hag i hi => hag i hi) (fun _ _ hag i hi => hag i hi)\n\n#print axioms ident3_decomposable", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data": "'ident3_decomposable' depends on axioms: [propext, Quot.sound]"}],
 "env": 5}

C'est la lecture la plus importante de la marche : **le théorème A s'applique dès que la localité est triviale** — l'identité n'a aucun couplage, chaque côté ne voit que lui. Le théorème ne dit rien en revanche des systèmes **non locaux** : la majorité du §2 et le broadcast du §8 mélangent tout le monde, et pour eux il faudra travailler. Notez aussi la continuité avec le §5 : la preuve de A **construit** le `z` que l'oracle cherchait par énumération — mais pour toute taille `n`, toute coupe, en un terme.

## 8. Teaser mesuré — le broadcast en taille 2

Le lake définit le **broadcast-parité** : `broadcast s = fun _ => parity n s` — l'état suivant est uniformément égal à la parité de l'état courant. Toute l'information globale est diffusée partout : c'est la forme la plus simple de « l'objet qui a mordu IIT », et l'objet des marches 2 et 3.

Sans toucher à sa preuve générale, mesurons-le en taille 2, où il n'existe qu'**une** coupe (`{0} | {1}`) — l'oracle peut fermer la question à lui seul.

In [7]:
def bits2 (s : State 2) : Bool × Bool := (s 0, s 1)

def cutHalf : Cut 2 where
  sel i := decide (i.val = 0)
  ha := ⟨0, by decide⟩
  hb := ⟨1, by decide⟩

-- Oracle en taille 2 :
def cutIndep2 (c : Cut 2) (f : System 2) : Bool :=
  (allStates 2).all fun x =>
    (allStates 2).all fun y =>
      (allStates 2).any fun z =>
        bits2 (projT c (f z)) == bits2 (projT c (f x))
        && bits2 (projF c (f z)) == bits2 (projF c (f y))

-- Les 4 etats, leur parite, l'etat suivant du broadcast :
#eval (allStates 2).map (fun s => (bits2 s, bits2 (broadcast (n := 2) s)))

-- La seule coupe possible est-elle independante ?
#eval cutIndep2 cutHalf (broadcast (n := 2))

def bits2 (s : State 2) : Bool × Bool := (s 0, s 1)

def cutHalf : Cut 2 where
  sel i := decide (i.val = 0)
  ha := ⟨0, by decide⟩
  hb := ⟨1, by decide⟩

-- Oracle en taille 2 :
def cutIndep2 (c : Cut 2) (f : System 2) : Bool :=
  (allStates 2).all fun x =>
    (allStates 2).all fun y =>
      (allStates 2).any fun z =>
        bits2 (projT c (f z)) == bits2 (projT c (f x))
        && bits2 (projF c (f z)) == bits2 (projF c (f y))

-- Les 4 etats, leur parite, l'etat suivant du broadcast :
#eval (allStates 2).map (fun s => (bits2 s, bits2 (broadcast (n := 2) s)))
─────▶  [((false, false), false, false), ((true, false), true, true), ((false, true), true, true), ((true, true), false, false)]

-- La seule coupe possible est-elle independante ?
#eval cutIndep2 cutHalf (broadcast (n := 2))
─────▶  false
--% env 6

Raw input:
{"cmd": "def bits2 (s : State 2) : Bool \u00d7 Bool := (s 0, s 1)\n\ndef cutHalf : Cut 2 where\n  sel i := decide (i.val = 0)\n  ha := \u27e80, by decide\u27e9\n  hb := \u27e81, by decide\u27e9\n\n-- Oracle en taille 2 :\ndef cutIndep2 (c : Cut 2) (f : System 2) : Bool :=\n  (allStates 2).all fun x =>\n    (allStates 2).all fun y =>\n      (allStates 2).any fun z =>\n        bits2 (projT c (f z)) == bits2 (projT c (f x))\n        && bits2 (projF c (f z)) == bits2 (projF c (f y))\n\n-- Les 4 etats, leur parite, l'etat suivant du broadcast :\n#eval (allStates 2).map (fun s => (bits2 s, bits2 (broadcast (n := 2) s)))\n\n-- La seule coupe possible est-elle independante ?\n#eval cutIndep2 cutHalf (broadcast (n := 2))", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 17, "column": 0},
   "endPos": {"line": 17, "column": 5},
   "data":
   "[((false, false), false, false), ((true, false), true, true), ((false, true), true, true), ((true, true), false, false)]"},
  {"severity": "info",
   "pos": {"line": 20, "column": 0},
   "endPos": {"line": 20, "column": 5},
   "data": "false"}],
 "env": 6}

L'image du broadcast vaut `{(false, false), (true, true)}` : deux états constants, la parité du départ. Chaque côté réalise séparément les deux motifs (`false` et `true`), donc le produit en compte quatre — mais les couples croisés `(A vrai, B faux)` et `(A faux, B vrai)` ne sont **jamais** réalisés ensemble : l'oracle rend `false`, la coupe n'est pas indépendante. À taille 2, le broadcast est **intégré** — par épuisement.

L'épuisement ne prouve rien au-delà : le nombre d'états double à chaque élément, le nombre de coupes aussi, et la question devient « existe-t-il un argument qui ferme **toutes** les coupes de **toutes** les tailles à la fois ? » — exactement le théorème B `broadcast_integrated` de la marche 2, dont la preuve instancie le couple croisé ci-dessus pour une coupe quelconque. Le notebook 06 donne l'autre moitié du portrait : le même broadcast, mesuré côté numérique sur des instances de plus en plus grandes.

## 9. Exercices

Préparation commune aux exercices 2 et 3 : un système à 4 éléments fait de deux moitiés **réellement différentes** — côté A (`{0, 1}`) un calcul conjonctif/disjonctif local, côté B (`{2, 3}`) une négation et une copie.

In [8]:
-- Systeme a 4 elements : cote A = {0,1}, cote B = {2,3}, aucun couplage.
def fLoc : System 4 := fun s i =>
  if i.val = 0 then s 0 && s 1
  else if i.val = 1 then s 0 || s 1
  else if i.val = 2 then ! (s 2)
  else s 3

def cut01 : Cut 4 where
  sel i := decide (i.val < 2)
  ha := ⟨0, by decide⟩
  hb := ⟨2, by decide⟩

def bits4 (s : State 4) : Bool × Bool × Bool × Bool := (s 0, s 1, s 2, s 3)

def cutIndep4 (c : Cut 4) (f : System 4) : Bool :=
  (allStates 4).all fun x =>
    (allStates 4).all fun y =>
      (allStates 4).any fun z =>
        bits4 (projT c (f z)) == bits4 (projT c (f x))
        && bits4 (projF c (f z)) == bits4 (projF c (f y))

#eval (allStates 4).take 4 |>.map (fun s => bits4 (fLoc s))

-- Systeme a 4 elements : cote A = {0,1}, cote B = {2,3}, aucun couplage.
def fLoc : System 4 := fun s i =>
  if i.val = 0 then s 0 && s 1
  else if i.val = 1 then s 0 || s 1
  else if i.val = 2 then ! (s 2)
  else s 3

def cut01 : Cut 4 where
  sel i := decide (i.val < 2)
  ha := ⟨0, by decide⟩
  hb := ⟨2, by decide⟩

def bits4 (s : State 4) : Bool × Bool × Bool × Bool := (s 0, s 1, s 2, s 3)

def cutIndep4 (c : Cut 4) (f : System 4) : Bool :=
  (allStates 4).all fun x =>
    (allStates 4).all fun y =>
      (allStates 4).any fun z =>
        bits4 (projT c (f z)) == bits4 (projT c (f x))
        && bits4 (projF c (f z)) == bits4 (projF c (f y))

#eval (allStates 4).take 4 |>.map (fun s => bits4 (fLoc s))
─────▶  [(false, false, true, false), (false, true, true, false), (false, true, true, false), (true, true, true, false)]
--% env 7

Raw input:
{"cmd": "-- Systeme a 4 elements : cote A = {0,1}, cote B = {2,3}, aucun couplage.\ndef fLoc : System 4 := fun s i =>\n  if i.val = 0 then s 0 && s 1\n  else if i.val = 1 then s 0 || s 1\n  else if i.val = 2 then ! (s 2)\n  else s 3\n\ndef cut01 : Cut 4 where\n  sel i := decide (i.val < 2)\n  ha := \u27e80, by decide\u27e9\n  hb := \u27e82, by decide\u27e9\n\ndef bits4 (s : State 4) : Bool \u00d7 Bool \u00d7 Bool \u00d7 Bool := (s 0, s 1, s 2, s 3)\n\ndef cutIndep4 (c : Cut 4) (f : System 4) : Bool :=\n  (allStates 4).all fun x =>\n    (allStates 4).all fun y =>\n      (allStates 4).any fun z =>\n        bits4 (projT c (f z)) == bits4 (projT c (f x))\n        && bits4 (projF c (f z)) == bits4 (projF c (f y))\n\n#eval (allStates 4).take 4 |>.map (fun s => bits4 (fLoc s))", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 22, "column": 0},
   "endPos": {"line": 22, "column": 5},
   "data":
   "[(false, false, true, false), (false, true, true, false), (false, true, true, false), (true, true, true, false)]"}],
 "env": 7}

### Exercice 1 — Une autre coupe, d'autres motifs

Sur le système à 3 éléments du §2, construisez la coupe `{1, 2} | {0}` (le complémentaire de `cutSolo`) et calculez les motifs de `sTrue` et `sMix` pour cette coupe.

*Étapes : (1) définir `cutPair : Cut 3` avec `sel i := decide (i.val ≠ 0)` — les témoins `ha`/`hb` se closent par `by decide` ; (2) évaluer `projT cutPair` et `projF cutPair` sur les deux états ; (3) vérifier que les motifs sont bien les complémentaires de ceux du §4 — même état, autre fenêtre.*

In [9]:
-- Exercice 1 : construire cutPair ({1,2} | {0}) et lire ses motifs.
-- Etape 1 : def cutPair : Cut 3 where ...
-- Etape 2 : #eval (bits3 (projT cutPair sTrue), bits3 (projF cutPair sTrue)) ...
-- Etape 3 : comparer avec la coupe {0} | {1,2} du 4.
#eval "Exercice 1 a completer"

-- Exercice 1 : construire cutPair ({1,2} | {0}) et lire ses motifs.
-- Etape 1 : def cutPair : Cut 3 where ...
-- Etape 2 : #eval (bits3 (projT cutPair sTrue), bits3 (projF cutPair sTrue)) ...
-- Etape 3 : comparer avec la coupe {0} | {1,2} du 4.
#eval "Exercice 1 a completer"
─────▶  "Exercice 1 a completer"
--% env 8

Raw input:
{"cmd": "-- Exercice 1 : construire cutPair ({1,2} | {0}) et lire ses motifs.\n-- Etape 1 : def cutPair : Cut 3 where ...\n-- Etape 2 : #eval (bits3 (projT cutPair sTrue), bits3 (projF cutPair sTrue)) ...\n-- Etape 3 : comparer avec la coupe {0} | {1,2} du 4.\n#eval \"Exercice 1 a completer\"", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "\"Exercice 1 a completer\""}],
 "env": 8}

### Exercice 2 — Appliquer le théorème A à `fLoc`

Prouvez `fLoc_decomposable : decomposable fLoc` en instanciant `product_decomposable` sur la coupe `cut01` — exactement comme le §7 le fait pour l'identité, mais cette fois la localité demande un vrai argument : les bits de sortie 0 et 1 ne dépendent que des bits d'entrée 0 et 1 (hypothèse `hA`), les bits 2 et 3 que des bits 2 et 3 (hypothèse `hB`).

*Étapes : (1) `refine product_decomposable cut01 fLoc ?_ ?_` ; (2) pour `hA` : intro des états et de l'accord côté A, puis disjonction sur `i.val = 0` ou `1` — chaque cas est une équation entre expressions qui ne lisent que `s 0` et `s 1` ; (3) pour `hB` : idem sur 2 et 3 ; (4) conclure avec `#print axioms`.*

*Indice : l'hypothèse d'accord `hag : ∀ i, cut01.sel i = true → s i = s' i` fournit `hag 0` et `hag 1` (les littéraux `Fin` se construisent par `⟨0, by omega⟩` ou directement `0`) ; le but après `simp [fLoc]` se réduit à réécrire avec ces deux égalités — `rw` ou `simp [*]` suffisent.*

In [10]:
-- Exercice 2 : theorem fLoc_decomposable : decomposable fLoc := by ...
-- Etape 1 : refine product_decomposable cut01 fLoc ?_ ?_
-- Etape 2 : hA -- les bits 0, 1 de sortie ne lisent que s 0, s 1
-- Etape 3 : hB -- les bits 2, 3 de sortie ne lisent que s 2, s 3
-- Etape 4 : #print axioms fLoc_decomposable
#eval "Exercice 2 a completer"

-- Exercice 2 : theorem fLoc_decomposable : decomposable fLoc := by ...
-- Etape 1 : refine product_decomposable cut01 fLoc ?_ ?_
-- Etape 2 : hA -- les bits 0, 1 de sortie ne lisent que s 0, s 1
-- Etape 3 : hB -- les bits 2, 3 de sortie ne lisent que s 2, s 3
-- Etape 4 : #print axioms fLoc_decomposable
#eval "Exercice 2 a completer"
─────▶  "Exercice 2 a completer"
--% env 9

Raw input:
{"cmd": "-- Exercice 2 : theorem fLoc_decomposable : decomposable fLoc := by ...\n-- Etape 1 : refine product_decomposable cut01 fLoc ?_ ?_\n-- Etape 2 : hA -- les bits 0, 1 de sortie ne lisent que s 0, s 1\n-- Etape 3 : hB -- les bits 2, 3 de sortie ne lisent que s 2, s 3\n-- Etape 4 : #print axioms fLoc_decomposable\n#eval \"Exercice 2 a completer\"", "env": 8}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 5},
   "data": "\"Exercice 2 a completer\""}],
 "env": 9}

### Exercice 3 — Quand le théorème A ne s'applique pas

La coupe `cut02 : {0, 2} | {1, 3}` **croise** les moitiés de `fLoc`. Montrez, par l'oracle du §9, que cette coupe n'est **pas** indépendante pour `fLoc` — puis expliquez pourquoi le théorème A ne dit rien ici (quel bit de sortie lit un bit de l'autre côté ?).

*Étapes : (1) définir `cut02 : Cut 4` avec `sel i := decide (i.val = 0 ∨ i.val = 2)` ; (2) évaluer `cutIndep4 cut02 fLoc` et lire `false` ; (3) identifier le bit de sortie qui viole la localité : le bit 0 vaut `s 0 && s 1`, or `s 1` est côté B ; (4) conclure : `fLoc` reste décomposable (exercice 2, via `cut01`) — un système peut être décomposable pour une coupe et insécable pour une autre, seule l'**existence** compte dans `decomposable`.*

*Indice : pour lire quelle paire de motifs manque, adaptez le listing du §8 — énumérez les couples `(bits4 (projT cut02 (fLoc z)), bits4 (projF cut02 (fLoc z)))` sur les 16 états et cherchez celui qui n'y figure pas.*

In [11]:
-- Exercice 3 : la coupe croisee {0,2} | {1,3} face a l'oracle.
-- Etape 1 : def cut02 : Cut 4 where sel i := decide (i.val = 0 || i.val = 2) ...
-- Etape 2 : #eval cutIndep4 cut02 fLoc        -- attendu : false
-- Etape 3 : quel bit de sortie lit l'autre cote ?
-- Etape 4 : relier a decomposable (existence, pas universalite).
#eval "Exercice 3 a completer"

-- Exercice 3 : la coupe croisee {0,2} | {1,3} face a l'oracle.
-- Etape 1 : def cut02 : Cut 4 where sel i := decide (i.val = 0 || i.val = 2) ...
-- Etape 2 : #eval cutIndep4 cut02 fLoc        -- attendu : false
-- Etape 3 : quel bit de sortie lit l'autre cote ?
-- Etape 4 : relier a decomposable (existence, pas universalite).
#eval "Exercice 3 a completer"
─────▶  "Exercice 3 a completer"
--% env 10

Raw input:
{"cmd": "-- Exercice 3 : la coupe croisee {0,2} | {1,3} face a l'oracle.\n-- Etape 1 : def cut02 : Cut 4 where sel i := decide (i.val = 0 || i.val = 2) ...\n-- Etape 2 : #eval cutIndep4 cut02 fLoc        -- attendu : false\n-- Etape 3 : quel bit de sortie lit l'autre cote ?\n-- Etape 4 : relier a decomposable (existence, pas universalite).\n#eval \"Exercice 3 a completer\"", "env": 9}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 5},
   "data": "\"Exercice 3 a completer\""}],
 "env": 10}

## 10. Ce que cette marche a établi — et ce qui manque

**Établi** : le vocabulaire complet de la coupe (état, système, coupe non triviale, motifs, indépendance comme produit d'images), le théorème A et son témoin constructif — le mélange des entrées — appliqué jusqu'au rendu par le noyau, et le piège des quantificateurs : décomposable se paie par un témoin, intégré exige de fermer toutes les coupes.

**Manque** : l'objet même de la veine. Le broadcast-parité est **intégré pour toute coupe** (théorème B, marche 2) **et de dynamique triviale** — toute orbite atteint un point fixe en deux pas (théorème C, capstone). L'oracle du §8 a fermé la taille 2 ; le théorème B ferme toutes les tailles et toutes les coupes d'un seul argument. Le capstone lira alors A, B et C **ensemble** : un système qu'aucune coupe ne dissout, dont le comportement ne fait rien — le contenu formel de la critique d'Aaronson, en deux théorèmes.

**Grade honnête** (rappelé du lake) : le proxy mesure la décomposition déterministe de l'image. Il ne capture ni l'information effective sous perturbation, ni la partition minimale au sens MIP, ni une valeur numérique de Φ — renvoyées à une Phase 2. La source de la critique est un billet de blog (2014), pas une publication formelle : le lake **crée** l'objet formel, il n'en est pas une transcription. Pour la moitié empirique de l'échange — Φ_DM sur F_p, le hack W, les grilles PyPhi — voir le [notebook 06](IIT-06-L-Objet-qui-a-Mordu-IIT-Python.ipynb) ; pour l'architecture d'ensemble des trois marches, la [note de conception](../../docs/lean/iit-lean-escalier.md).

## 11. Références

| Objet | Où |
| --- | --- |
| Lake source (19 déclarations, 0 `sorry`, sans Mathlib) | [`SymbolicAI/Lean/iit_lean`](../SymbolicAI/Lean/iit_lean/README.md) — livré par #18599 |
| Note de conception de l'escalier (3 marches) | [`docs/lean/iit-lean-escalier.md`](../../docs/lean/iit-lean-escalier.md) — #20414 |
| Moitié empirique de la veine (Φ_DM, PyPhi) | [`IIT-06-L-Objet-qui-a-Mordu-IIT-Python`](IIT-06-L-Objet-qui-a-Mordu-IIT-Python.ipynb) |
| Epic de rattachement (veine 5) | #16781 — « l'objet qui a mordu IIT, en Lean » |
| L'échange source (mai 2014) | cité et gradué dans le notebook 06, §Le débat — billet de blog, pas une publication |